# Week 3: Train Contrastive Probe - IMPROVED VERSION

## Three Key Improvements

### 1. Code Mass Threshold (35%)
Instead of requiring >50% code votes, we stop if ≥35% of probability mass is code tokens.

### 2. MLP Classifier
Replaced LogisticRegression with MLPClassifier for better non-linear decision boundaries.

### 3. Active Learning
Added pure language prompts to training data, labeled as LANGUAGE even if they generate code artifacts (backticks, markdown, etc.).

---

In [ ]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import entropy as scipy_entropy
from sklearn.neural_network import MLPClassifier  # ✨ IMPROVEMENT 2: MLP instead of LogisticRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports complete")

In [ ]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    output_hidden_states=True
)
model.eval()

print(f"✅ Model loaded on {model.device}")

## Step 1: Generate Active Learning Data

**IMPROVEMENT 3**: Generate pure language prompts and label ALL tokens as LANGUAGE

In [ ]:
# Cell 4: Generate Active Learning Data

PURE_LANGUAGE_PROMPTS = [
    # Nature & Weather
    "The color of the sky is",
    "The weather today feels",
    "The sunset looked",
    "The rain is",
    "The flowers in the garden are",
    
    # Personal & Emotions
    "I love eating",
    "My favorite hobby is",
    "Yesterday I felt",
    "The best part of my day was",
    "I really enjoy",
    
    # Stories & Books
    "The book I read last week was",
    "The story begins with",
    "The main character in the novel is",
    "The ending of the movie was",
    "My favorite book is",
    
    # Daily Life
    "The meeting yesterday was",
    "My morning routine includes",
    "The coffee tastes",
    "The restaurant we visited had",
    "My weekend plans include",
    
    # Descriptions
    "The house on the corner is",
    "The cat sitting on the windowsill looks",
    "The music playing in the background is",
    "The painting on the wall depicts",
    "The view from the window shows",
]

def softmax(logits: np.ndarray) -> np.ndarray:
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def get_top_k_tokens(prompt: str, k: int = 20) -> list:
    """Get top-K candidate next tokens with their probabilities."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :].cpu().numpy()
    
    probs = softmax(logits)
    top_indices = np.argsort(probs)[-k:][::-1]
    
    candidates = []
    for idx in top_indices:
        token = tokenizer.decode([idx])
        prob = float(probs[idx])
        candidates.append({
            'token': token,
            'probability': prob,
        })
    
    return candidates

print("\n🔄 Generating active learning data...")
active_learning_data = []

for prompt in tqdm(PURE_LANGUAGE_PROMPTS, desc="Pure language prompts"):
    candidates = get_top_k_tokens(prompt, k=20)
    
    for candidate in candidates:
        full_text = prompt + candidate['token']
        
        active_learning_data.append({
            'prompt': prompt,
            'expected_type': 'language',
            'next_token': candidate['token'],
            'probability': candidate['probability'],
            'full_text': full_text,
            'label': 'language'  # ✨ ALL labeled as language, even code artifacts
        })

df_active = pd.DataFrame(active_learning_data)
print(f"\n✅ Generated {len(df_active)} active learning examples")

# Check for code-like tokens
code_artifacts = df_active[df_active['next_token'].str.contains(r'[`#\[\]{}()<>]', regex=True, na=False)]
print(f"   Code-like artifacts (labeled as LANGUAGE): {len(code_artifacts)}")
if len(code_artifacts) > 0:
    print(f"\n   Examples:")
    print(code_artifacts[['prompt', 'next_token']].head(5))

In [ ]:
# Cell 5: Load and Combine Data

# Option 1: Upload your existing training_data_labeled.csv to Colab
# Option 2: Load from Google Drive
# For now, we'll assume it's uploaded to /content/

try:
    LABELED_DATA_FILE = '/content/training_data_labeled.csv'
    df_existing = pd.read_csv(LABELED_DATA_FILE)
    print(f"✅ Loaded {len(df_existing)} existing training examples")
    
    # Combine
    df = pd.concat([df_existing, df_active], ignore_index=True)
    print(f"✅ Combined: {len(df)} total examples")
    
except FileNotFoundError:
    print("⚠️  Original training data not found. Using only active learning data.")
    df = df_active

# Validate and convert labels
valid_labels = df['label'].isin(['code', 'language'])
if not valid_labels.all():
    print(f"\n⚠️  Removing {(~valid_labels).sum()} rows with invalid labels")
    df = df[valid_labels]

df['label_binary'] = df['label'].map({'language': 0, 'code': 1})

print(f"\n📊 Final Training Data:")
print(f"   Total: {len(df)}")
print(f"   LANGUAGE (0): {(df['label_binary']==0).sum()}")
print(f"   CODE (1): {(df['label_binary']==1).sum()}")
print(f"\nSample:")
print(df[['full_text', 'label', 'probability']].head(10))

In [ ]:
# Cell 6: Extract Hidden States

SELECTED_LAYERS = [8, 16, 31]

def get_multi_layer_state(text: str, layers: List[int]) -> np.ndarray:
    """Extract and concatenate hidden states from multiple layers."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    states = [
        outputs.hidden_states[layer_idx + 1][:, -1, :].cpu().numpy()[0]
        for layer_idx in layers
    ]
    return np.concatenate(states).astype(np.float32)

print(f"Extracting hidden states from layers {SELECTED_LAYERS}...")
print(f"This may take a few minutes for {len(df)} examples...\n")

X_train = []
y_train = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
    h = get_multi_layer_state(row['full_text'], SELECTED_LAYERS)
    X_train.append(h)
    y_train.append(row['label_binary'])

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"\n✅ Hidden states extracted")
print(f"   Shape: {X_train.shape}")
print(f"   Feature dimension: {X_train.shape[1]:,} (3 layers × 4096)")

## Step 2: Train MLP Classifier

**IMPROVEMENT 2**: Using MLPClassifier instead of LogisticRegression

In [ ]:
# Cell 7: Train MLP Probe

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# ✨ IMPROVEMENT 2: MLP Classifier
probe = MLPClassifier(
    hidden_layer_sizes=(128, 64),  # Two hidden layers
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42,
    alpha=0.0001,  # L2 regularization
    early_stopping=True,  # Stop when validation score doesn't improve
    validation_fraction=0.1,
    n_iter_no_change=10,
    verbose=False
)

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(probe, X_train_scaled, y_train, cv=cv)

cv_accuracy = accuracy_score(y_train, y_pred_cv)

print(f"\n{'='*80}")
print(f"MLP PROBE TRAINING RESULTS")
print(f"{'='*80}")
print(f"\nArchitecture: MLPClassifier(hidden_layers=[128, 64])")
print(f"Layers: {SELECTED_LAYERS}")
print(f"Training examples: {len(y_train)}")
print(f"5-Fold CV Accuracy: {cv_accuracy:.1%}")

print(f"\n{classification_report(y_train, y_pred_cv, target_names=['LANGUAGE (0)', 'CODE (1)'])}")

# Confusion matrix
cm = confusion_matrix(y_train, y_pred_cv)
print(f"Confusion Matrix:")
print(f"                Pred LANG  Pred CODE")
print(f"True LANG          {cm[0,0]:>4}       {cm[0,1]:>4}")
print(f"True CODE          {cm[1,0]:>4}       {cm[1,1]:>4}")

# Train final model on all data
probe.fit(X_train_scaled, y_train)
print(f"\n✅ Final MLP probe trained on all data")
print(f"   Training iterations: {probe.n_iter_}")
print(f"   Final loss: {probe.loss_:.4f}")

## Step 3: Contrastive Generation with Code Mass Threshold

**IMPROVEMENT 1**: Using CODE_MASS_THRESHOLD = 0.35 instead of >50%

In [ ]:
# Cell 8: Helper Functions

def entropy_from_probs(probs: np.ndarray) -> float:
    if len(probs) == 0 or np.sum(probs) == 0:
        return 0.0
    norm_probs = probs / np.sum(probs)
    return float(scipy_entropy(norm_probs, base=2))

def classify_token_type(text: str) -> Tuple[int, float]:
    """Classify: 1=code token, 0=language token."""
    h = get_multi_layer_state(text, SELECTED_LAYERS).reshape(1, -1)
    h_scaled = scaler.transform(h)
    token_type = probe.predict(h_scaled)[0]
    probability = probe.predict_proba(h_scaled)[0, 1]
    return int(token_type), float(probability)

# ✨ IMPROVEMENT 1: Code Mass Threshold
CODE_MASS_THRESHOLD = 0.35  # Stop if ≥35% probability mass is CODE tokens

def analyze_candidate_tokens(
    prompt: str,
    top_k: int = 10,
    verbose: bool = False
) -> Dict:
    """
    Get top-K candidate next tokens and classify each as CODE or LANGUAGE.
    IMPROVED: Use CODE_MASS_THRESHOLD instead of majority vote.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :].cpu().numpy()

    probs = softmax(logits)
    top_indices = np.argsort(probs)[-top_k:][::-1]

    candidates = []
    code_votes = 0
    lang_votes = 0

    for idx in top_indices:
        token = tokenizer.decode([idx])
        prob = probs[idx]

        # Classify prompt + candidate token
        completion = prompt + token
        token_type, type_prob = classify_token_type(completion)

        candidates.append({
            'token': token,
            'prob': prob,
            'type': 'CODE' if token_type == 1 else 'LANGUAGE',
            'type_prob': type_prob
        })

        if token_type == 1:
            code_votes += prob
        else:
            lang_votes += prob

    # ✨ IMPROVEMENT 1: Threshold-based decision
    is_code_uncertainty = code_votes > CODE_MASS_THRESHOLD

    if verbose:
        print(f"\nCandidate analysis:")
        for c in candidates:
            print(f"  '{c['token']}' (p={c['prob']:.3f}) → {c['type']} (conf={c['type_prob']:.3f})")
        print(f"\nVotes: CODE={code_votes:.3f}, LANGUAGE={lang_votes:.3f}")
        print(f"Threshold: {CODE_MASS_THRESHOLD}")
        print(f"Decision: {'CODE uncertainty - STOP' if is_code_uncertainty else 'LANGUAGE uncertainty - CONTINUE'}")

    return {
        'candidates': candidates,
        'code_votes': code_votes,
        'lang_votes': lang_votes,
        'is_code_uncertainty': is_code_uncertainty,
        'code_mass_ratio': code_votes / (code_votes + lang_votes) if (code_votes + lang_votes) > 0 else 0
    }

print(f"✅ Helper functions ready")
print(f"   CODE_MASS_THRESHOLD = {CODE_MASS_THRESHOLD}")

In [ ]:
# Cell 9: Contrastive Generation Function

def generate_with_contrastive_probe(
    prompt: str,
    entropy_threshold: float = 3.0,
    top_k_candidates: int = 10,
    max_tokens: int = 50,
    verbose: bool = True
) -> Dict:
    """
    Contrastive entropy-driven generation with IMPROVED decision logic.
    
    At each step:
    1. Generate next token
    2. Compute entropy H
    3. If H > threshold:
       - Get top-K candidate next tokens
       - Classify each "prompt + candidate" as CODE or LANGUAGE
       - Weighted vote: If code_votes > CODE_MASS_THRESHOLD → STOP
       - Otherwise → Continue
    4. If H ≤ threshold: Continue (confident)
    """
    current_text = prompt
    generated_token_ids = []
    entropy_trace = []
    stop_reason = None
    stop_info = {}

    if verbose:
        print(f"\n{'='*80}")
        print(f"Prompt: '{prompt}'")
        print(f"Entropy threshold: {entropy_threshold:.1f} bits")
        print(f"Code mass threshold: {CODE_MASS_THRESHOLD}")
        print(f"{'='*80}")

    for step in range(max_tokens):
        inputs = tokenizer(current_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits[0, -1, :].cpu().numpy()

        probs = softmax(logits)
        H = entropy_from_probs(probs)
        entropy_trace.append(H)

        next_token_id = np.argmax(probs)
        next_token = tokenizer.decode([next_token_id])

        if verbose:
            print(f"\nStep {step + 1}: '{next_token}' H={H:.2f}")

        if H > entropy_threshold:
            if verbose:
                print(f"  ⚠️  HIGH ENTROPY - analyzing candidates...")

            analysis = analyze_candidate_tokens(
                current_text,
                top_k=top_k_candidates,
                verbose=verbose
            )

            if analysis['is_code_uncertainty']:
                if verbose:
                    print(f"  ❗ CODE UNCERTAINTY - STOPPING!")
                    print(f"     Code mass: {analysis['code_votes']:.3f} > {CODE_MASS_THRESHOLD}")
                stop_reason = "code_uncertainty"
                stop_info = {
                    'step': step,
                    'entropy': H,
                    'code_votes': analysis['code_votes'],
                    'lang_votes': analysis['lang_votes'],
                    'code_mass_ratio': analysis['code_mass_ratio']
                }
                break
            else:
                if verbose:
                    print(f"  ✓ LANGUAGE uncertainty - continuing")
                    print(f"     Code mass: {analysis['code_votes']:.3f} ≤ {CODE_MASS_THRESHOLD}")

        generated_token_ids.append(next_token_id)
        current_text += next_token

        if next_token_id == tokenizer.eos_token_id:
            stop_reason = "eos"
            break

    if stop_reason is None:
        stop_reason = "max_tokens"

    generated_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True)

    if verbose:
        print(f"\n{'='*80}")
        print(f"Stop: {stop_reason}")
        print(f"Generated: '{generated_text}'")
        print(f"Full: '{prompt}{generated_text}'")
        print(f"{'='*80}")

    return {
        'prompt': prompt,
        'generated_text': generated_text,
        'full_text': prompt + generated_text,
        'entropy_trace': entropy_trace,
        'stop_reason': stop_reason,
        'stop_info': stop_info,
        'num_steps': len(generated_token_ids)
    }

print("✅ Generation function ready")

## Testing

In [ ]:
# Cell 10: Demo Test

print("\n" + "="*80)
print("DEMO: Testing IMPROVED Contrastive Probe")
print("="*80)

test_prompts = [
    "The authentication is done using",  # Should STOP (code uncertainty)
    "The book I read last week was",     # Should CONTINUE (pure language)
]

for test_prompt in test_prompts:
    result = generate_with_contrastive_probe(
        test_prompt,
        entropy_threshold=3.0,
        top_k_candidates=10,
        max_tokens=10,
        verbose=True
    )
    print("\n" + "-"*80 + "\n")

In [ ]:
# Cell 11: Full Test Suite

CODE_TEST_CASES = [
    {'prompt': 'In our React app, authentication is done using', 'category': 'auth_method'},
    {'prompt': 'In the backend, passwords are hashed with', 'category': 'auth_hash'},
    {'prompt': 'For our API, JWT tokens are signed using', 'category': 'auth_signing'},
    {'prompt': 'In production, the OAuth provider we use is', 'category': 'auth_provider'},
    {'prompt': 'On the server, session data is stored in', 'category': 'session_store'},
    {'prompt': 'For data persistence, the database we use is', 'category': 'database_type'},
    {'prompt': 'In the application, we query the database using', 'category': 'database_query'},
    {'prompt': 'For database access, the ORM library is', 'category': 'database_orm'},
    {'prompt': 'To improve performance, caching is implemented with', 'category': 'database_cache'},
    {'prompt': 'For the REST API, the framework we use is', 'category': 'web_framework'},
    {'prompt': 'In production, the web server runs on', 'category': 'web_server'},
    {'prompt': 'In the client code, HTTP requests are made using', 'category': 'http_client'},
    {'prompt': 'For data fetching, our GraphQL server uses', 'category': 'graphql_server'},
    {'prompt': 'For the UI, the frontend framework is', 'category': 'frontend_framework'},
    {'prompt': 'In the application, state management is handled by', 'category': 'frontend_state'},
    {'prompt': 'For the interface, components are built with', 'category': 'frontend_components'},
    {'prompt': 'In the SPA, routing is done using', 'category': 'frontend_routing'},
    {'prompt': 'For training, the model is trained with', 'category': 'ml_framework'},
    {'prompt': 'In our neural network, deep learning is implemented using', 'category': 'ml_deep_learning'},
    {'prompt': 'For gradient descent, the optimizer we use is', 'category': 'ml_optimizer'},
    {'prompt': 'For hosting, we deploy to', 'category': 'cloud_platform'},
    {'prompt': 'In Kubernetes, containers are orchestrated with', 'category': 'cloud_containers'},
    {'prompt': 'For automation, the CI/CD pipeline uses', 'category': 'cloud_cicd'},
    {'prompt': 'In the test suite, unit tests are written with', 'category': 'test_unit'},
    {'prompt': 'For building assets, the bundler we use is', 'category': 'build_bundler'},
    {'prompt': 'For dependencies, package management is done with', 'category': 'build_package_manager'},
]

LANGUAGE_TEST_CASES = [
    {'prompt': 'The weather today is', 'category': 'description_weather'},
    {'prompt': 'The meeting yesterday was', 'category': 'description_meeting'},
    {'prompt': 'My favorite color has always been', 'category': 'description_color'},
    {'prompt': 'The book I read last week was', 'category': 'description_book'},
    {'prompt': 'The movie we watched seemed', 'category': 'description_movie'},
    {'prompt': 'The main idea of the story is to', 'category': 'explanation_idea'},
    {'prompt': 'The cooking process works by', 'category': 'explanation_process'},
    {'prompt': 'This teaching approach helps to', 'category': 'explanation_approach'},
    {'prompt': 'The benefit of exercise is', 'category': 'explanation_benefit'},
    {'prompt': 'When installing furniture in my home, you should', 'category': 'instruction_furniture'},
    {'prompt': 'To debug a relationship problem, first', 'category': 'instruction_debug'},
    {'prompt': 'Before deploying troops, the general needs to', 'category': 'instruction_deploy'},
    {'prompt': 'The configuration of the room requires', 'category': 'instruction_config'},
    {'prompt': 'To optimize your morning routine, try', 'category': 'instruction_optimize'},
    {'prompt': 'The framework of the argument is', 'category': 'nontechnical_framework'},
    {'prompt': 'My mental state is managed by', 'category': 'nontechnical_state'},
    {'prompt': 'The library in town has', 'category': 'nontechnical_library'},
    {'prompt': 'Running the business takes', 'category': 'nontechnical_running'},
    {'prompt': 'The function of the heart is', 'category': 'nontechnical_function'},
    {'prompt': 'Implementing the new policy will', 'category': 'nontechnical_implement'},
]

ALL_TEST_CASES = [
    {**case, 'expected_stopped': True} for case in CODE_TEST_CASES
] + [
    {**case, 'expected_stopped': False} for case in LANGUAGE_TEST_CASES
]

print(f"\n{'='*80}")
print(f"FULL TEST SUITE - IMPROVED VERSION")
print(f"{'='*80}")
print(f"\nTotal test cases: {len(ALL_TEST_CASES)}")
print(f"  CODE tests (should stop): {len(CODE_TEST_CASES)}")
print(f"  LANGUAGE tests (should not stop): {len(LANGUAGE_TEST_CASES)}")

print(f"\n🔄 Running tests...\n")

test_results = []

for test_case in tqdm(ALL_TEST_CASES, desc="Testing"):
    prompt = test_case['prompt']
    
    result = generate_with_contrastive_probe(
        prompt,
        entropy_threshold=3.0,
        top_k_candidates=10,
        max_tokens=20,
        verbose=False
    )
    
    stopped_for_code = result['stop_reason'] == 'code_uncertainty'
    expected_stopped = test_case['expected_stopped']
    correct = stopped_for_code == expected_stopped
    
    max_entropy = np.max(result['entropy_trace']) if result['entropy_trace'] else 0
    
    test_results.append({
        'prompt': prompt,
        'category': test_case['category'],
        'expected_stopped': expected_stopped,
        'stopped_for_code': stopped_for_code,
        'correct': correct,
        'stop_reason': result['stop_reason'],
        'max_entropy': max_entropy,
        'generated_text': result.get('generated_text', ''),
        'num_tokens_generated': result.get('num_steps', 0),
        'code_mass_ratio': result.get('stop_info', {}).get('code_mass_ratio', 0)
    })

print(f"\n✅ Testing complete!")

# Analysis
df_results = pd.DataFrame(test_results)

print(f"\n{'='*80}")
print(f"IMPROVED RESULTS")
print(f"{'='*80}")

overall_accuracy = df_results['correct'].mean()
print(f"\n📊 OVERALL ACCURACY: {overall_accuracy:.1%}")
print(f"   (on {len(df_results)} test cases)")

print(f"\n📈 BY CLASS:")
for expected_val in [True, False]:
    class_name = "CODE (should stop)" if expected_val else "LANGUAGE (should not stop)"
    subset = df_results[df_results['expected_stopped'] == expected_val]
    accuracy = subset['correct'].mean() if len(subset) > 0 else 0
    correct = subset['correct'].sum()
    total = len(subset)
    stopped_count = subset['stopped_for_code'].sum()
    print(f"\n   {class_name}")
    print(f"      Correct: {correct}/{total}")
    print(f"      Accuracy: {accuracy:.1%}")
    print(f"      Stopped for code: {stopped_count}/{total}")
    if total > 0:
        print(f"      Avg entropy: {subset['max_entropy'].mean():.2f} bits")
        print(f"      Avg tokens generated: {subset['num_tokens_generated'].mean():.1f}")

# Confusion matrix
tp = len(df_results[(df_results['expected_stopped']==True) & (df_results['stopped_for_code']==True)])
fp = len(df_results[(df_results['expected_stopped']==False) & (df_results['stopped_for_code']==True)])
fn = len(df_results[(df_results['expected_stopped']==True) & (df_results['stopped_for_code']==False)])
tn = len(df_results[(df_results['expected_stopped']==False) & (df_results['stopped_for_code']==False)])

print(f"\n🎯 CONFUSION MATRIX")
print(f"                       Predicted NO STOP    Predicted STOPPED")
print(f"Expected NO STOP             {tn:<12}          {fp:<12}")
print(f"Expected STOP                {fn:<12}          {tp:<12}")

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n📐 METRICS (CODE class - should stop)")
print(f"   Precision: {precision:.1%}")
print(f"   Recall: {recall:.1%}")
print(f"   F1-Score: {f1:.1%}")

# Errors
errors = df_results[~df_results['correct']]
if len(errors) > 0:
    print(f"\n❌ ERRORS ({len(errors)}/{len(df_results)}):")
    for idx, error in errors.iterrows():
        exp = "SHOULD STOP" if error['expected_stopped'] else "SHOULD NOT STOP"
        act = "STOPPED" if error['stopped_for_code'] else "DID NOT STOP"
        print(f"\n   Prompt: '{error['prompt']}'")
        print(f"      Expected: {exp}, Actual: {act}")
        print(f"      Generated: '{error['generated_text'][:60]}...'")
        print(f"      Stop reason: {error['stop_reason']}, Max entropy: {error['max_entropy']:.2f}")
else:
    print(f"\n🎉 PERFECT! No errors!")

print(f"\n{'='*80}")
print(f"IMPROVEMENTS SUMMARY")
print(f"{'='*80}")
print(f"✨ 1. CODE_MASS_THRESHOLD = {CODE_MASS_THRESHOLD} (instead of >50%)")
print(f"✨ 2. MLP Classifier with [128, 64] hidden layers")
print(f"✨ 3. Active learning: {len(df_active)} pure language examples added")
print(f"\n📈 Expected improvement: Better recall on CODE cases, fewer false positives on LANGUAGE cases")
print(f"{'='*80}")